# 04 - ML dengan Spark MLlib
Jalan di Colab (CPU High-RAM, tanpa GPU). Fitur & label identik dengan notebook 03 supaya perbandingan sklearn vs MLlib adil.

In [ ]:
# Jalankan hanya di Colab / environment dengan Java tersedia
!pip install -q pyspark


In [ ]:
import os, sys, time
import pandas as pd
from pyspark.sql import SparkSession

BASE_DIR = os.path.abspath(os.environ.get("BDA_BASE_DIR", "/content/big-data-aol"))
print("BASE_DIR aktif:", BASE_DIR)
assert os.path.isdir(os.path.join(BASE_DIR, "src")), (
    f"BASE_DIR salah: {BASE_DIR} tidak punya folder src/. "
    "Set os.environ['BDA_BASE_DIR'] ke path repo yang benar sebelum run cell ini."
)
assert os.path.isfile(os.path.join(BASE_DIR, "data", "processed", "earthquake_features.parquet")), (
    "earthquake_features.parquet belum ada di BASE_DIR ini. "
    "Jalankan 02_cleaning.ipynb dulu dengan BDA_BASE_DIR yang SAMA persis sebelum notebook ini."
)

PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
FIG_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

sys.path.insert(0, os.path.join(BASE_DIR, "src"))
import features as ft

spark = (
    SparkSession.builder
    .appName("bda-earthquake-risk")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark


In [ ]:
# Label + fitur SAMA PERSIS kayak notebook 03: dibangun di pandas (fungsi src/features.py
# dipakai bersama, bukan ditulis ulang di Spark) supaya definisi risk_label dan
# FEATURE_COLUMNS identik. Baru dikonversi ke Spark DataFrame setelah siap.
#
# risk_label = risiko zona pada horizon 90 hari KE DEPAN; fitur dari masa lalu/saat ini.
pdf = pd.read_parquet(os.path.join(PROCESSED_DIR, "earthquake_features.parquet"))
pdf = ft.add_ml_features(pdf)
pdf = ft.add_risk_label(pdf)
print("Threshold label:", pdf.attrs["risk_label_thresholds"])
print(pdf["risk_label"].value_counts(normalize=True).round(4))

# Keluarkan kejadian tanpa jendela horizon penuh, samakan dengan notebook 03
pdf = pdf[pdf["has_full_horizon"]]
print("Baris setelah filter horizon:", len(pdf))

cols = ft.FEATURE_COLUMNS + ft.CATEGORICAL_COLUMNS + [ft.TARGET_COLUMN, "year"]
sdf = spark.createDataFrame(pdf[cols])
sdf.printSchema()


In [ ]:
# Split TEMPORAL sama - train <=2022, test 2023-2026
train_sdf = sdf.filter(sdf.year <= 2022)
test_sdf = sdf.filter(sdf.year > 2022)
print("train:", train_sdf.count(), "| test:", test_sdf.count())


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier

label_indexer = StringIndexer(inputCol=ft.TARGET_COLUMN, outputCol="label")
cat_indexer = StringIndexer(inputCol="mag_type", outputCol="mag_type_idx", handleInvalid="keep")
cat_encoder = OneHotEncoder(inputCol="mag_type_idx", outputCol="mag_type_vec")

num_assembler = VectorAssembler(inputCols=ft.FEATURE_COLUMNS, outputCol="num_features")
scaler = StandardScaler(inputCol="num_features", outputCol="num_features_scaled")

final_assembler = VectorAssembler(
    inputCols=["num_features_scaled", "mag_type_vec"], outputCol="features"
)

rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)

pipeline = Pipeline(stages=[
    label_indexer, cat_indexer, cat_encoder,
    num_assembler, scaler, final_assembler, rf,
])


In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [100, 200])
    .addGrid(rf.maxDepth, [8, 12])
    .build()
)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

cv = CrossValidator(
    estimator=pipeline, estimatorParamMaps=param_grid,
    evaluator=evaluator, numFolds=3, seed=42,
)


In [ ]:
t0 = time.time()
cv_model = cv.fit(train_sdf)
fit_time_spark = time.time() - t0
print(f"Spark MLlib CrossValidator fit time: {fit_time_spark:.2f}s")

best_model = cv_model.bestModel
best_rf = best_model.stages[-1]
print("Best numTrees:", best_rf.getNumTrees, "| Best maxDepth:", best_rf.getMaxDepth())


In [ ]:
predictions = best_model.transform(test_sdf)

# Catat mapping index -> label asli (StringIndexer urutkan berdasar frekuensi, bukan alfabet)
label_mapping = dict(enumerate(best_model.stages[0].labelsArray[0]))
print("Label mapping (index -> risk_label):", label_mapping)

# PENTING: MulticlassClassificationEvaluator Spark memakai skema WEIGHTED,
# bukan macro. metricName="f1" pada Spark = weighted F1. Bukti: weightedRecall
# selalu sama persis dengan accuracy. Karena itu kolom di bawah diberi akhiran
# _weighted, dan pembandingnya di notebook 03 juga memakai skema weighted --
# membandingkan weighted (Spark) dengan macro (sklearn) akan menyesatkan.
acc_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
precision_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_eval = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

spark_metrics = dict(
    model="spark_mllib_random_forest",
    accuracy=round(acc_eval.evaluate(predictions), 4),
    f1_weighted=round(f1_eval.evaluate(predictions), 4),
    precision_weighted=round(precision_eval.evaluate(predictions), 4),
    recall_weighted=round(recall_eval.evaluate(predictions), 4),
    f1_macro=None,       # MLlib tidak menyediakan skema macro secara bawaan
    precision_macro=None,
    recall_macro=None,
    roc_auc_ovr=None,    # MLlib tidak menyediakan AUC one-vs-rest secara bawaan
    fit_time_s=round(fit_time_spark, 3),
)
spark_metrics


In [ ]:
# Bandingkan dengan hasil sklearn (notebook 03).
# Perbandingan utama memakai skema WEIGHTED di kedua sisi supaya setara,
# karena Spark MLlib hanya menyediakan skema weighted.
sklearn_metrics_path = os.path.join(PROCESSED_DIR, "metrics_sklearn.json")
sklearn_metrics = pd.read_json(sklearn_metrics_path)

spark_row = pd.DataFrame([spark_metrics])
comparison = pd.concat([sklearn_metrics, spark_row], ignore_index=True)

kolom = ["model", "accuracy", "f1_weighted", "precision_weighted", "recall_weighted",
         "f1_macro", "roc_auc_ovr", "fit_time_s"]
comparison = comparison[[k for k in kolom if k in comparison.columns]]
print("Kolom *_weighted sebanding lintas pustaka; f1_macro & AUC hanya tersedia di sklearn.")
comparison


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

comparison.set_index("model")[["accuracy", "f1_weighted"]].plot(kind="bar", ax=axes[0])
axes[0].set_title("Akurasi & F1 Weighted (skema setara lintas pustaka)")
axes[0].set_ylim(0, 1.0)
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(loc="lower right", fontsize=8)

comparison.set_index("model")["fit_time_s"].plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Waktu Training (detik, skala log)")
axes[1].set_yscale("log")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "04_sklearn_vs_spark.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
comparison.to_json(os.path.join(PROCESSED_DIR, "metrics_spark.json"), orient="records")
print("Tersimpan: metrics_spark.json")
spark.stop()


In [ ]:
# Push hasil (metrics json, figures) ke GitHub supaya bisa dicek tanpa Drive
sys.path.insert(0, os.path.join(BASE_DIR, "src"))
import sync
sync.push_results(BASE_DIR, message="Update hasil 04_ml_spark")
